In [ ]:
!pip install hqq
!pip install transformers datasets huggingface_hub
!pip install wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
import os
import numpy as np
import pandas as pd
user_secrets = UserSecretsClient()
token=user_secrets.get_secret("HUGGINGFACE_API_KEY")
secret_value_1 = user_secrets.get_secret("WANDB_LOGIN_API")
os.environ["WANDB_API_KEY"] = secret_value_1
from huggingface_hub import login

login(token=token)
# Log in to WandB
wandb.login()

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HqqConfig
)
import torch, multiprocessing

model_id = "microsoft/Phi-3.5-mini-instruct"
# All linear layers will use the same quantization config
quant_config = HqqConfig(nbits=1, group_size=64, quant_zero=False, quant_scale=False, axis=1)

#Load the tokenizer to save it along with the model
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token)
tokenizer.pad_token = tokenizer.eos_token

# Load and quantize
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=quant_config,
    token=token
)
from hqq.models.hf.base import AutoHQQHFModel

#Save: Make sure to save the model BEFORE any patching
AutoHQQHFModel.save_quantized(model, "./Phi-3.5-mini-instruct_HQQ-1bit/")



In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HqqConfig
)
import torch, multiprocessing

model_id = "microsoft/Phi-3.5-mini-instruct"
# All linear layers will use the same quantization config
quant_config = HqqConfig(nbits=4, group_size=64, quant_zero=False, quant_scale=False, axis=1)

#Load the tokenizer to save it along with the model
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token)
tokenizer.pad_token = tokenizer.eos_token

# Load and quantize
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=quant_config,
    token=token
)


from hqq.models.hf.base import AutoHQQHFModel

#Save: Make sure to save the model BEFORE any patching
AutoHQQHFModel.save_quantized(model, "./Phi-3.5-mini-instruct_HQQ-4bit/")



In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HqqConfig
)
import torch, multiprocessing

model_id = "meta-llama/Meta-Llama-3-8B"
# All linear layers will use the same quantization config
quant_config = HqqConfig(nbits=2, group_size=64, quant_zero=False, quant_scale=False, axis=1)

#Load the tokenizer to save it along with the model
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token)
tokenizer.pad_token = tokenizer.eos_token

# Load and quantize
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=quant_config,
    token=token
)

from hqq.models.hf.base import AutoHQQHFModel

#Save: Make sure to save the model BEFORE any patching
AutoHQQHFModel.save_quantized(model, "./Phi-3.5-mini-instruct_HQQ-2bit/")



In [ ]:
import os

def get_dir_size_mb(path):
    total_bytes = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 * 1024)

# (approx. param count, baseline dtype bytes/param, saved quantized checkpoint dir)
# Note: the 2-bit Llama-3-8B checkpoint above is saved under the
# "Phi-3.5-mini-instruct_HQQ-2bit" folder name (copy/paste path from the Phi cells).
checkpoints = {
    "Phi-3.5-mini-instruct (1-bit HQQ)": (3.82e9, 2, "./Phi-3.5-mini-instruct_HQQ-1bit/"),
    "Phi-3.5-mini-instruct (4-bit HQQ)": (3.82e9, 2, "./Phi-3.5-mini-instruct_HQQ-4bit/"),
    "Meta-Llama-3-8B (2-bit HQQ)": (8.03e9, 2, "./Phi-3.5-mini-instruct_HQQ-2bit/"),
}

print(f"{'Checkpoint':<38}{'Baseline (GB)':>15}{'Quantized (GB)':>16}{'Reduction':>12}")
footprint_results = {}
for name, (n_params, dtype_bytes, path) in checkpoints.items():
    baseline_mb = (n_params * dtype_bytes) / (1024 * 1024)
    quant_mb = get_dir_size_mb(path)
    reduction_pct = ((baseline_mb - quant_mb) / baseline_mb) * 100
    footprint_results[name] = {"baseline_mb": baseline_mb, "quantized_mb": quant_mb, "reduction_pct": reduction_pct}
    print(f"{name:<38}{baseline_mb / 1024:>15.2f}{quant_mb / 1024:>16.2f}{reduction_pct:>11.2f}%")

if wandb.run is not None:
    for name, r in footprint_results.items():
        wandb.log({f"footprint/{name}/reduction_pct": r["reduction_pct"]})

In [ ]:
import torch
from datasets import load_dataset
from tqdm import tqdm

def calculate_perplexity(model, tokenizer, test_dataset, max_length=512, stride=256):
    model.eval()
    device = next(model.parameters()).device

    encodings = tokenizer("\n\n".join(test_dataset["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0

    for begin_loc in tqdm(range(0, seq_len, stride), desc="Evaluating Perplexity"):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / seq_len)
    return ppl.item()

# Evaluates whichever quantized model is currently loaded in memory
# (the Meta-Llama-3-8B 2-bit checkpoint from the cell above, at this point in the notebook)
test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ppl_score = calculate_perplexity(model, tokenizer, test_data)
print(f"Quantized Model Perplexity (WikiText-2): {ppl_score:.2f}")

if wandb.run is not None:
    wandb.log({"eval/perplexity_wikitext2": ppl_score})

In [ ]:
import time

def benchmark_inference(model, tokenizer, prompt="Explain quantum computing in simple terms:", max_new_tokens=64):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Warm-up run
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
        )

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_time = time.time() - start_time

    generated_tokens = output.shape[1] - inputs.input_ids.shape[1]
    throughput = generated_tokens / elapsed_time
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if device.type == "cuda" else 0.0

    print(f"Generated text: {tokenizer.decode(output[0], skip_special_tokens=True)}")
    print("-" * 50)
    print(f"Tokens Generated     : {generated_tokens}")
    print(f"Elapsed Time         : {elapsed_time:.3f} s")
    print(f"Inference Throughput : {throughput:.2f} tokens/s")
    print(f"Peak VRAM Usage      : {peak_vram_mb:.2f} MB")

    if wandb.run is not None:
        wandb.log({
            "latency/throughput_tokens_per_sec": throughput,
            "latency/peak_vram_mb": peak_vram_mb,
        })

# Benchmarks whichever quantized model is currently loaded in memory
# (the Meta-Llama-3-8B 2-bit checkpoint from the cells above, at this point in the notebook)
benchmark_inference(model, tokenizer)